# CRISP SMILES Experiment Runner — Runpod

This notebook runs all 12 manuscript experiments (6 ZINC translation + 6 USPTO reaction prediction) with 3 seeds each.

**Requirements**: GPU instance with `transformers`, `torch`, `rdkit` installed.

## Setup

In [ ]:
# Clone the repo and install dependencies
# !git clone https://github.com/denovochem/crisp_smiles.git
# %cd crisp_smiles
# !pip install -r requirements.txt
# !pip install pytest ruff mypy

In [ ]:
import sys
sys.path.insert(0, "/workspace/crisp_smiles")

# Verify imports work
from crisp_smiles.main import CRISPSmiles
from models.tokenizer.vocab import smiles_token_to_id_dict
from models.tokenizer.tokenizer import CustomTokenizer
from models.training.trainer import SmilesTrainer, CustomSmallConfig, compute_top_tokens, strip_stereo_from_smiles

print(f"Vocabulary size: {len(smiles_token_to_id_dict)}")
print(f"[STEREO_E] ID: {smiles_token_to_id_dict['[STEREO_E]']}")
print(f"[STEREO_Z] ID: {smiles_token_to_id_dict['[STEREO_Z]']}")
print(f"[DB_STEREO_1] ID: {smiles_token_to_id_dict['[DB_STEREO_1]']}")
print(f"[UTILITY_TOKEN_26] ID: {smiles_token_to_id_dict['[UTILITY_TOKEN_26]']}")

## 2. Prepare Datasets

Upload `zinc250k.txt` and USPTO files to `/data/` on the Runpod instance.

In [ ]:
import os

# Verify dataset paths
zinc_path = "/data/zinc250k.txt"
uspto_dir = "/data"

assert os.path.exists(zinc_path), f"ZINC dataset not found at {zinc_path}"
assert os.path.exists(os.path.join(uspto_dir, "JIN_USPTO_1product_train.txt")), "USPTO train not found"

with open(zinc_path) as f:
    zinc_lines = [l.strip() for l in f if l.strip()]
print(f"ZINC: {len(zinc_lines)} molecules")
print(f"First: {zinc_lines[0]}")

## 3. Run All Experiments

This runs 12 experiments × 3 seeds = 36 training runs total.
Each run trains for 50,000 steps with evaluation every 5,000 steps.

In [ ]:
# Run all experiments via the CLI
# This will take many hours on a single GPU
# Results are saved to results/ directory

# !python scripts/generate_manuscript_data.py \
#     --zinc-smiles-path /data/zinc250k.txt \
#     --uspto-dir /data \
#     --seeds 42 123 456

## 4. Run a Single Experiment (for testing)

Use this to verify the pipeline works before launching the full suite.

In [ ]:
from scripts.generate_manuscript_data import (
    get_default_experiments,
    run_experiment,
    _read_lines,
)
from pathlib import Path

# Load a small subset for quick testing
zinc_smiles = _read_lines(Path("/data/zinc250k.txt"))
zinc_train = zinc_smiles[:1000]
zinc_val = zinc_smiles[1000:1200]

experiments = get_default_experiments()
print(f"Total experiments: {len(experiments)}")
for i, exp in enumerate(experiments):
    print(f"  {i+1}. {exp.dataset} / {exp.task}")

In [ ]:
# Run just the first experiment (ZINC SMILES→SMILES) with a small subset
exp = experiments[0]
print(f"Running: {exp.dataset} / {exp.task}")

# Override max_steps for quick testing
from models.training.trainer import CustomSmallConfig
quick_config = CustomSmallConfig(
    **{**exp.config.__dict__, "max_steps": 100, "eval_steps": 50, "save_steps": 50}
)
quick_exp = type(exp)(
    dataset=exp.dataset,
    task=exp.task,
    config=quick_config,
    input_preprocessing=exp.input_preprocessing,
    output_preprocessing=exp.output_preprocessing,
)
results = run_experiment(quick_exp, zinc_train, zinc_val)
print(f"Results: {results}")

## 5. Analyze Results

After experiments complete, use the stereochem flip analysis script to examine per-atom stereo flips.

In [ ]:
# Analyze stereochem flips for a specific eval generation file
# Replace with the actual path to your eval_generations file
# from scripts.analyze_stereochem_flips import analyze_file
# from pathlib import Path
#
# gen_path = Path("results/zinc_smiles_to_smiles/seed_42/analysis/eval_generations_step_50000.txt")
# analyze_file(gen_path, output_path=Path("results/zinc_smiles_to_smiles/seed_42/stereo_flips.txt"))

# Or run via CLI:
# !python scripts/analyze_stereochem_flips.py \
#     results/zinc_smiles_to_smiles/seed_42/analysis/eval_generations_step_50000.txt \
#     --output results/zinc_smiles_to_smiles/seed_42/stereo_flips.txt

## 6. Collect and Summarize Results

Read `seed_summary.txt` files from each experiment directory to compare conditions.

In [ ]:
import pandas as pd
from pathlib import Path

# Collect all seed summaries
results_dir = Path("results")
summaries = []
for summary_path in sorted(results_dir.glob("*/seed_summary.txt")):
    df = pd.read_csv(summary_path, sep="\\t")
    df["experiment"] = summary_path.parent.name
    summaries.append(df)

if summaries:
    all_results = pd.concat(summaries, ignore_index=True)
    # Compute mean ± std per experiment
    agg = all_results.groupby("experiment").agg(["mean", "std"])
    print(agg)
    
    # Save consolidated summary
    all_results.to_csv(results_dir / "all_results.csv", index=False)
    agg.to_csv(results_dir / "aggregated_summary.csv")
    print("Saved to results/all_results.csv and results/aggregated_summary.csv")
else:
    print("No seed_summary.txt files found yet. Run experiments first.")